In [17]:
import ccxt
import pandas as pd
import numpy as np
from datetime import datetime as dt
#import matplotlib.pyplot as plt
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Exploración de Datos Históricos de Binance con CCXT

Este notebook explora los datos históricos de criptomonedas de Binance utilizando la librería CCXT para análisis de mercado.

## 1. Configuración de Conexión con Deribit

In [12]:
# Inicializar la conexión con Deribit
exchange = ccxt.deribit({
    'apiKey': '',  # No necesario para datos públicos
    'secret': '',  # No necesario para datos públicos
    'sandbox': False,  # Usar datos reales de producción
    'enableRateLimit': True,  # Habilitar límite de velocidad
})

# Verificar la conexión
print(f"Exchange: {exchange.name}")
print(f"Has fetch_ohlcv: {exchange.has['fetchOHLCV']}")
print(f"Timeframes disponibles: {list(exchange.timeframes.keys())}")

Exchange: Deribit
Has fetch_ohlcv: True
Timeframes disponibles: ['1m', '3m', '5m', '10m', '15m', '30m', '1h', '2h', '3h', '6h', '12h', '1d']


## 2. Obtener Mercados de Trading Disponibles

In [48]:
# Cargar información de mercados
markets = exchange.load_markets()

# Convertir a DataFrame para mejor visualización
markets_df = pd.DataFrame.from_dict(markets, orient='index', dtype=object)
markets_df['created'] = pd.to_datetime(markets_df['created'], unit='ms', errors='coerce')
markets_df['expiry'] = pd.to_datetime(markets_df['expiry'], unit='ms', errors='coerce')

# Mostrar información básica
print(f"Total de mercados disponibles: {len(markets_df)}")
print(f"Columnas disponibles: {list(markets_df.columns)}")
print(f"Created min: {markets_df['created'].min()}, max: {markets_df['created'].max()}")
print(f"Expiry min: {markets_df['expiry'].min()}, max: {markets_df['expiry'].max()}")


Total de mercados disponibles: 4570
Columnas disponibles: ['id', 'lowercaseId', 'symbol', 'base', 'quote', 'settle', 'baseId', 'quoteId', 'settleId', 'type', 'spot', 'margin', 'swap', 'future', 'option', 'index', 'active', 'contract', 'linear', 'inverse', 'subType', 'taker', 'maker', 'contractSize', 'expiry', 'expiryDatetime', 'strike', 'optionType', 'precision', 'limits', 'marginModes', 'created', 'info', 'tierBased', 'percentage']
Created min: 2018-08-14 10:24:47, max: 2025-10-26 09:51:50
Expiry min: 2025-10-27 08:00:00, max: 2026-09-25 08:00:00


### 2.1 Mercados de opciones
#### Filtrar solo mercados de opciones activos

In [50]:
option_markets = markets_df[
    (markets_df['option'] == True) & 
    (markets_df['base'] == 'BTC') & 
    (markets_df['quote'] == 'USD')
    #(markets_df['created'] > dt(2025, 1, 1)) # Opciones creadas después de enero 2025
    #(markets_df['expiry'] < dt(2025, 2, 28))  # Opciones expiradas hasta febrero 2025
]
print(f"Total de mercados de opciones BTC/USD: {len(option_markets)}")

option_markets.head(10)

Total de mercados de opciones BTC/USD: 728


,id,lowercaseId,symbol,base,quote,settle,baseId,quoteId,settleId,type,...,expiryDatetime,strike,optionType,precision,limits,marginModes,created,info,tierBased,percentage
BTC/USD:BTC-251027-102000-C,BTC-27OCT25-102000-C,None,BTC/USD:BTC-251027-102000-C,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,102000.0,call,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-102000-P,BTC-27OCT25-102000-P,None,BTC/USD:BTC-251027-102000-P,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,102000.0,put,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-104000-C,BTC-27OCT25-104000-C,None,BTC/USD:BTC-251027-104000-C,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,104000.0,call,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-104000-P,BTC-27OCT25-104000-P,None,BTC/USD:BTC-251027-104000-P,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,104000.0,put,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-105000-C,BTC-27OCT25-105000-C,None,BTC/USD:BTC-251027-105000-C,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,105000.0,call,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-105000-P,BTC-27OCT25-105000-P,None,BTC/USD:BTC-251027-105000-P,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,105000.0,put,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-106000-C,BTC-27OCT25-106000-C,None,BTC/USD:BTC-251027-106000-C,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,106000.0,call,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-106000-P,BTC-27OCT25-106000-P,None,BTC/USD:BTC-251027-106000-P,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,106000.0,put,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-107000-C,BTC-27OCT25-107000-C,None,BTC/USD:BTC-251027-107000-C,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,107000.0,call,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None
BTC/USD:BTC-251027-107000-P,BTC-27OCT25-107000-P,None,BTC/USD:BTC-251027-107000-P,BTC,USD,BTC,BTC,USD,BTC,option,...,2025-10-27T08:00:00.000Z,107000.0,put,"{'amount': 0.1, 'price': 0.0001, 'cost': None,...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-23 08:00:12,"{'price_index': 'btc_usd', 'kind': 'option', '...",None,None


### 2.2 Marcados de futuros
#### filtrar mercado de fufutros perpetuos

In [51]:
future_markets = markets_df[
    (markets_df['future'] == True) &
    (markets_df['base'] == 'BTC') &
    (markets_df['quote'] == 'USD')
]
print(f"Numero de mercado des futuros BTC/USD: {len(future_markets)}")
future_markets

Numero de mercado des futuros BTC/USD: 35


,id,lowercaseId,symbol,base,quote,settle,baseId,quoteId,settleId,type,...,expiryDatetime,strike,optionType,precision,limits,marginModes,created,info,tierBased,percentage
BTC/USD:BTC-251031,BTC-31OCT25,None,BTC/USD:BTC-251031,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-10-31T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 2.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-08-29 08:00:19,"{'price_index': 'btc_usd', 'kind': 'future', '...",None,None
BTC-FS-26DEC25_31OCT25,BTC-FS-26DEC25_31OCT25,None,BTC-FS-26DEC25_31OCT25,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-10-31T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 0.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-08-29 08:07:21,"{'price_index': 'btc_usd', 'kind': 'future_com...",None,None
BTC-FS-27MAR26_31OCT25,BTC-FS-27MAR26_31OCT25,None,BTC-FS-27MAR26_31OCT25,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-10-31T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 0.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-08-29 08:53:13,"{'price_index': 'btc_usd', 'kind': 'future_com...",None,None
BTC-FS-26JUN26_31OCT25,BTC-FS-26JUN26_31OCT25,None,BTC-FS-26JUN26_31OCT25,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-10-31T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 0.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-08-29 08:53:13,"{'price_index': 'btc_usd', 'kind': 'future_com...",None,None
BTC-FS-31OCT25_PERP,BTC-FS-31OCT25_PERP,None,BTC-FS-31OCT25_PERP,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-10-31T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 0.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-08-29 08:53:13,"{'price_index': 'btc_usd', 'kind': 'future_com...",None,None
BTC-FS-28NOV25_31OCT25,BTC-FS-28NOV25_31OCT25,None,BTC-FS-28NOV25_31OCT25,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-10-31T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 0.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-09-26 09:41:47,"{'price_index': 'btc_usd', 'kind': 'future_com...",None,None
BTC-FS-25SEP26_31OCT25,BTC-FS-25SEP26_31OCT25,None,BTC-FS-25SEP26_31OCT25,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-10-31T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 0.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-09-26 12:19:25,"{'price_index': 'btc_usd', 'kind': 'future_com...",None,None
BTC-FS-7NOV25_31OCT25,BTC-FS-7NOV25_31OCT25,None,BTC-FS-7NOV25_31OCT25,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-10-31T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 0.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-24 08:00:15,"{'price_index': 'btc_usd', 'kind': 'future_com...",None,None
BTC/USD:BTC-251107,BTC-7NOV25,None,BTC/USD:BTC-251107,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-11-07T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 2.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-24 08:00:15,"{'price_index': 'btc_usd', 'kind': 'future', '...",None,None
BTC-FS-25SEP26_7NOV25,BTC-FS-25SEP26_7NOV25,None,BTC-FS-25SEP26_7NOV25,BTC,USD,BTC,BTC,USD,BTC,future,...,2025-11-07T08:00:00.000Z,NaN,None,"{'amount': 10.0, 'price': 0.5, 'cost': None, '...","{'leverage': {'min': None, 'max': None}, 'amou...","{'cross': None, 'isolated': None}",2025-10-24 08:00:15,"{'price_index': 'btc_usd', 'kind': 'future_com...",None,None


In [44]:
markets_df['settle'].unique()

array([None, 'USDC', 'BTC', 'ETH'], dtype=object)

## 3. Descargar Datos Históricos OHLCV

In [47]:
# Función para obtener datos históricos
def fetch_ohlcv_data(symbol, timeframe='1d', limit=100):
    """
    Obtiene datos OHLCV históricos para un símbolo específico
    """
    try:
        ohlcv = exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
        df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
        return df
    except Exception as e:
        print(f"Error obteniendo datos para {symbol}: {e}")
        return None

# Obtener datos para varios pares principales
symbols = ['btc_usdt', 'eth_usdt', 'bnb_usdt']
historical_data = {}

for symbol in symbols:
    print(f"Obteniendo datos para {symbol}...")
    data = fetch_ohlcv_data(symbol, timeframe='1d', limit=365)  # Último año
    if data is not None:
        historical_data[symbol] = data
        print(f"✓ Obtenidos {len(data)} registros para {symbol}")
    else:
        print(f"✗ Error obteniendo datos para {symbol}")

# Mostrar muestra de datos para BTC/USDT
if 'BTC/USDT' in historical_data:
    print(f"\nMuestra de datos BTC/USDT:")
    print(historical_data['BTC/USDT'].head())

Obteniendo datos para btc_usdt...
Error obteniendo datos para btc_usdt: deribit does not have market symbol btc_usdt
✗ Error obteniendo datos para btc_usdt
Obteniendo datos para eth_usdt...
Error obteniendo datos para eth_usdt: deribit does not have market symbol eth_usdt
✗ Error obteniendo datos para eth_usdt
Obteniendo datos para bnb_usdt...
Error obteniendo datos para bnb_usdt: deribit does not have market symbol bnb_usdt
✗ Error obteniendo datos para bnb_usdt


## 4. Preprocesamiento y Limpieza de Datos

In [ ]:
def preprocess_data(df):
    """
    Preprocesa los datos OHLCV
    """
    # Copiar DataFrame
    df_clean = df.copy()
    
    # Establecer datetime como índice
    df_clean.set_index('datetime', inplace=True)
    
    # Eliminar duplicados
    df_clean = df_clean.drop_duplicates()
    
    # Ordenar por fecha
    df_clean = df_clean.sort_index()
    
    # Calcular retornos diarios
    df_clean['returns'] = df_clean['close'].pct_change()
    
    # Calcular precio promedio
    df_clean['avg_price'] = (df_clean['high'] + df_clean['low']) / 2
    
    # Calcular rango diario
    df_clean['daily_range'] = df_clean['high'] - df_clean['low']
    
    # Calcular volatilidad (rango relativo)
    df_clean['volatility'] = df_clean['daily_range'] / df_clean['close']
    
    return df_clean

# Aplicar preprocesamiento a todos los datos
processed_data = {}
for symbol, data in historical_data.items():
    processed_data[symbol] = preprocess_data(data)
    print(f"Datos procesados para {symbol}: {len(processed_data[symbol])} registros")
    
# Mostrar estadísticas básicas
if 'BTC/USDT' in processed_data:
    print(f"\nInformación de datos BTC/USDT:")
    print(processed_data['BTC/USDT'].info())

## 5. Análisis Estadístico Básico

In [ ]:
def calculate_statistics(df, symbol):
    """
    Calcula estadísticas básicas para los datos de precio
    """
    stats = {
        'Symbol': symbol,
        'Period': f"{df.index.min().date()} to {df.index.max().date()}",
        'Days': len(df),
        'Avg_Price': df['close'].mean(),
        'Min_Price': df['close'].min(),
        'Max_Price': df['close'].max(),
        'Std_Price': df['close'].std(),
        'Avg_Volume': df['volume'].mean(),
        'Avg_Daily_Return': df['returns'].mean() * 100,
        'Return_Volatility': df['returns'].std() * 100,
        'Max_Daily_Gain': df['returns'].max() * 100,
        'Max_Daily_Loss': df['returns'].min() * 100,
    }
    return stats

# Calcular estadísticas para todos los símbolos
stats_summary = []
for symbol, data in processed_data.items():
    stats = calculate_statistics(data, symbol)
    stats_summary.append(stats)

# Crear DataFrame con estadísticas
stats_df = pd.DataFrame(stats_summary)
print("Resumen Estadístico:")
print(stats_df.round(4))

# Análisis de correlación entre símbolos
if len(processed_data) > 1:
    # Crear DataFrame con precios de cierre
    prices_df = pd.DataFrame()
    for symbol, data in processed_data.items():
        prices_df[symbol] = data['close']
    
    # Calcular correlaciones
    correlation_matrix = prices_df.corr()
    print(f"\nMatriz de Correlación de Precios:")
    print(correlation_matrix.round(4))

## 6. Visualización de Series Temporales

In [ ]:
def create_candlestick_chart(df, symbol):
    """
    Crea un gráfico de velas japonesas
    """
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=(f'{symbol} - Precio', 'Volumen'),
        row_width=[0.7, 0.3]
    )
    
    # Gráfico de velas
    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='Precio'
        ),
        row=1, col=1
    )
    
    # Gráfico de volumen
    fig.add_trace(
        go.Bar(
            x=df.index,
            y=df['volume'],
            name='Volumen',
            marker_color='rgba(158,202,225,0.8)'
        ),
        row=2, col=1
    )
    
    fig.update_layout(
        title=f'Análisis de Precio y Volumen - {symbol}',
        yaxis_title='Precio (USDT)',
        yaxis2_title='Volumen',
        xaxis_rangeslider_visible=False,
        height=700
    )
    
    return fig

# Crear gráficos para cada símbolo
for symbol, data in processed_data.items():
    fig = create_candlestick_chart(data, symbol)
    fig.show()
    
    # También crear un gráfico simple con matplotlib
    plt.figure(figsize=(12, 6))
    plt.plot(data.index, data['close'], linewidth=2, label=f'{symbol} Close Price')
    plt.title(f'{symbol} - Evolución del Precio')
    plt.xlabel('Fecha')
    plt.ylabel('Precio (USDT)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Análisis de Volumen

In [ ]:
def analyze_volume_patterns(df, symbol):
    """
    Analiza patrones de volumen y su relación con movimientos de precio
    """
    # Calcular volumen promedio móvil
    df['volume_ma_7'] = df['volume'].rolling(window=7).mean()
    df['volume_ma_30'] = df['volume'].rolling(window=30).mean()
    
    # Identificar días de alto volumen (> 2 desviaciones estándar)
    volume_threshold = df['volume'].mean() + 2 * df['volume'].std()
    df['high_volume'] = df['volume'] > volume_threshold
    
    # Analizar relación volumen-precio
    high_volume_days = df[df['high_volume']]
    
    stats = {
        'Symbol': symbol,
        'Avg_Volume': df['volume'].mean(),
        'High_Volume_Days': len(high_volume_days),
        'Avg_Return_High_Volume': high_volume_days['returns'].mean() * 100 if len(high_volume_days) > 0 else 0,
        'Avg_Return_Normal_Volume': df[~df['high_volume']]['returns'].mean() * 100,
        'Volume_Price_Correlation': df['volume'].corr(df['close'])
    }
    
    return stats, df

# Analizar volumen para todos los símbolos
volume_stats = []
for symbol, data in processed_data.items():
    stats, enhanced_data = analyze_volume_patterns(data, symbol)
    volume_stats.append(stats)
    processed_data[symbol] = enhanced_data  # Actualizar con nuevas columnas

volume_df = pd.DataFrame(volume_stats)
print("Análisis de Patrones de Volumen:")
print(volume_df.round(4))

# Crear visualización de volumen vs precio para BTC
if 'BTC/USDT' in processed_data:
    btc_data = processed_data['BTC/USDT']
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Precio
    ax1.plot(btc_data.index, btc_data['close'], label='BTC Precio', color='orange')
    ax1.set_ylabel('Precio (USDT)')
    ax1.set_title('BTC/USDT - Precio y Volumen')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Volumen
    colors = ['red' if x else 'blue' for x in btc_data['high_volume']]
    ax2.bar(btc_data.index, btc_data['volume'], color=colors, alpha=0.6, width=1)
    ax2.plot(btc_data.index, btc_data['volume_ma_30'], label='Volume MA 30', color='black', linewidth=2)
    ax2.set_ylabel('Volumen')
    ax2.set_xlabel('Fecha')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Patrones de Movimiento de Precios

In [ ]:
def analyze_price_patterns(df, symbol):
    """
    Analiza patrones de movimiento de precios y calcula indicadores técnicos básicos
    """
    # Medias móviles
    df['sma_7'] = df['close'].rolling(window=7).mean()
    df['sma_30'] = df['close'].rolling(window=30).mean()
    df['sma_90'] = df['close'].rolling(window=90).mean()
    
    # Volatilidad móvil
    df['volatility_30'] = df['returns'].rolling(window=30).std() * 100
    
    # Tendencia (pendiente de la media móvil de 30 días)
    df['trend'] = df['sma_30'].diff()
    
    # Identificar tendencias alcistas/bajistas
    df['bullish_trend'] = (df['close'] > df['sma_30']) & (df['trend'] > 0)
    df['bearish_trend'] = (df['close'] < df['sma_30']) & (df['trend'] < 0)
    
    # Calcular estadísticas de tendencias
    bullish_days = len(df[df['bullish_trend']])
    bearish_days = len(df[df['bearish_trend']])
    
    # Retornos acumulados
    df['cumulative_returns'] = (1 + df['returns']).cumprod() - 1
    
    # Drawdown (pérdida máxima)
    rolling_max = df['close'].expanding().max()
    drawdown = (df['close'] - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    stats = {
        'Symbol': symbol,
        'Bullish_Days': bullish_days,
        'Bearish_Days': bearish_days,
        'Trend_Ratio': bullish_days / bearish_days if bearish_days > 0 else float('inf'),
        'Total_Return': df['cumulative_returns'].iloc[-1] * 100,
        'Max_Drawdown': max_drawdown * 100,
        'Avg_Volatility': df['volatility_30'].mean(),
        'Current_vs_SMA30': (df['close'].iloc[-1] / df['sma_30'].iloc[-1] - 1) * 100
    }
    
    return stats, df

# Analizar patrones para todos los símbolos
pattern_stats = []
for symbol, data in processed_data.items():
    stats, enhanced_data = analyze_price_patterns(data, symbol)
    pattern_stats.append(stats)
    processed_data[symbol] = enhanced_data

pattern_df = pd.DataFrame(pattern_stats)
print("Análisis de Patrones de Precio:")
print(pattern_df.round(4))

# Visualización de tendencias para BTC
if 'BTC/USDT' in processed_data:
    btc_data = processed_data['BTC/USDT'].tail(90)  # Últimos 90 días
    
    plt.figure(figsize=(15, 10))
    
    # Subplot 1: Precio y medias móviles
    plt.subplot(3, 1, 1)
    plt.plot(btc_data.index, btc_data['close'], label='BTC Precio', linewidth=2, color='orange')
    plt.plot(btc_data.index, btc_data['sma_7'], label='SMA 7', alpha=0.8)
    plt.plot(btc_data.index, btc_data['sma_30'], label='SMA 30', alpha=0.8)
    plt.title('BTC/USDT - Precio y Medias Móviles (Últimos 90 días)')
    plt.ylabel('Precio (USDT)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Subplot 2: Volatilidad
    plt.subplot(3, 1, 2)
    plt.plot(btc_data.index, btc_data['volatility_30'], label='Volatilidad 30d', color='red')
    plt.ylabel('Volatilidad (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Subplot 3: Retornos acumulados
    plt.subplot(3, 1, 3)
    plt.plot(btc_data.index, btc_data['cumulative_returns'] * 100, label='Retornos Acumulados', color='green')
    plt.ylabel('Retornos Acumulados (%)')
    plt.xlabel('Fecha')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Resumen final
print(f"\n{'='*50}")
print("RESUMEN DEL ANÁLISIS")
print(f"{'='*50}")
print(f"Período analizado: {len(processed_data)} símbolos")
print(f"Datos desde: {list(processed_data.values())[0].index.min().date()}")
print(f"Datos hasta: {list(processed_data.values())[0].index.max().date()}")
print(f"\nDatos disponibles para análisis gamma-neutral y estrategias de trading.")